In [ ]:
import pandas as pd

df = pd.read_json("data/llm_hallucination_detection_dataset.jsonl", lines=True)
df.shape

In [ ]:
df.head(3)

In [ ]:
df["label"].value_counts()

In [ ]:
import re

def overlap(response, context):
    response_words = set(re.findall(r"[a-z0-9]+", str(response).lower()))
    context_words = set(re.findall(r"[a-z0-9]+", str(context).lower()))
    if not response_words:
        return 0
    return len(response_words & context_words) / len(response_words)

df["overlap"] = [overlap(r, c) for r, c in zip(df["response"], df["context"])]
df.groupby("label")["overlap"].mean()

In [ ]:
df["prediction"] = df["overlap"].apply(lambda x: "hallucinated" if x < 0.6 else "faithful")
(df["prediction"] == df["label"]).mean()

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)
(test["prediction"] == test["label"]).mean()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test["label"], test["prediction"]))

In [ ]:
misses = test[(test["label"] == "hallucinated") & (test["prediction"] == "faithful")]
print(len(misses))
misses[["prompt", "context", "response", "overlap"]].head(5)

The missed answers repeat most of the context and change one fact, such as a city, a date, or a number. The dataset also repeats the same examples, so those rows show up more than once.

In [ ]:
unique = df.drop_duplicates(["prompt", "context", "response"])
print(len(unique))

train_u, test_u = train_test_split(unique, test_size=0.2, random_state=42)
pred_u = test_u["overlap"].apply(lambda x: "hallucinated" if x < 0.6 else "faithful")
print((pred_u == test_u["label"]).mean())
print(classification_report(test_u["label"], pred_u))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

train_text = train_u["prompt"] + " " + train_u["context"] + " " + train_u["response"]
test_text = test_u["prompt"] + " " + test_u["context"] + " " + test_u["response"]

model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
model.fit(train_text, train_u["label"])
print(classification_report(test_u["label"], model.predict(test_text)))

In [ ]:
def novel_words(response, context):
    response_words = set(re.findall(r"[a-z0-9]+", str(response).lower()))
    context_words = set(re.findall(r"[a-z0-9]+", str(context).lower()))
    return " ".join(sorted(response_words - context_words))

train_novel = [novel_words(r, c) for r, c in zip(train_u["response"], train_u["context"])]
test_novel = [novel_words(r, c) for r, c in zip(test_u["response"], test_u["context"])]

model2 = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
model2.fit(train_novel, train_u["label"])
print(classification_report(test_u["label"], model2.predict(test_novel)))